This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences, savgol_filter

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide

### Functions

In [ ]:
# phi_flat = phi.counts.reshape(-1)
# phi_mids = phi.midpoints[1]
def get_graphable_data(phi: NDHistogram) -> tuple[np.ndarray, np.ndarray]:
    """Get data from phi in graphable format.

    Counts and energy bin midpoints will be extracted as 1D arrays.
    :param phi: Neutron energy spectrum
    :type phi: NDHistogram
    :return: Counts array, energy bin midpoints array
    :rtype: tuple[np.ndarray, np.ndarray]
    """
    return phi.counts.reshape(-1), phi.midpoints[-1]

## Setting Entry

In [ ]:
energies = np.arange(0.124, 6.014, step=0.062)
exp_data = {f"{energy:.3f}_MeV": {} for energy in energies}

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
# bins_max value determined by detector energy calibration range
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

## Data Processing

### Loading

In [ ]:
R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    Path("response_matrix_R4_mono"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
R.counts.shape

In [ ]:
np.sum(R.counts, axis=0)

In [ ]:
# widths = [15, 15, 20, 15, 15, 15, 15, 15]
bins = np.arange(bins_min, bins_max + bins_width, bins_width)

# base_path = Path("response_matrix_4k")
base_path = Path("response_matrix_R4_mono")
# TODO load all files in list
for energy, eng_data in exp_data.items():
    filepath = base_path / f"neutron_{energy}.csv.npy"
    eng_data["filepath"] = filepath

    # df = pd.read_fwf(filepath, widths=widths)

    # bins = np.arange(bins_min, bins_max + bins_width, bins_width)
    # cut = pd.cut(df["det_pulse (MeVee)"], bins.tolist())
    # cut_index = cut.cat.categories
    # new_df = pd.DataFrame(
    #     df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)
    # )

    # old_index = new_df.index
    # if not isinstance(old_index, pd.IntervalIndex):
    #     raise RuntimeError(
    #         f"DataFrame created from cut/groupby for {test_file.name} "
    #         + "was not an IntervalIndex as expected"
    #     )
    # mids = old_index.mid.to_series(index=cut_index)

    # np_cps = new_df["NPS"].to_numpy().reshape(-1, 1)
    # np_Ls = mids.to_numpy()

    # N = NDHistogram(np_cps, [np_Ls, np.ones(1)])

    L_array = np.load(filepath)
    
    np_cps, *_ = np.histogram(L_array, bins=bins)
    
    mov_avg_window = 7
    polyorder = 3
    np_cps_savgol = savgol_filter(np_cps, window_length=mov_avg_window, polyorder=polyorder)
    
    np_cps = np_cps.reshape(-1, 1)
    np_cps_savgol = np_cps_savgol.reshape(-1, 1)
    np_Ls = (bins[1:] + bins[:-1]) / 2
    
    if energy == "1.302_MeV":
    #     print(L_array)
        # print(np_cps)
    #     print(np_Ls)
        # print(np_cps_savgol)
        pass
    
    # N = NDHistogram(np_cps, [np_Ls, np.ones(1)])
    N = NDHistogram(np_cps_savgol, [np_Ls, np.ones(1)])
    eng_data["N"] = N

### Processing

In [ ]:
# if should_normalize:
#     R_max = R.counts.max(axis=0, keepdims=True)
#     N_max = N.counts.max()
#     norm_factor = _nan_divide(N_max, R_max)
#     N_norm = N.counts * norm_factor
#     N = NDHistogram(N_norm, R.midpoints)

In [ ]:
# if make_phi0:
#     R_L_mids, R_E_mids = R.midpoints
#     reduced_L_mids = np.array([R_L_mids.mean()])
#     if sim_type == SimType.TWOFOURFIVE:  # 2.45 MeV
#         phi0_counts = np.ones_like(R_E_mids)
#         phi0_counts = phi0_counts.reshape(1, -1)
#         peak_idx = np.argmax(R_E_mids >= 2.45)
#         phi0_counts[0, peak_idx] = 100
#     elif sim_type == SimType.AMBE:  # AmBe
#         phi0_counts = np.interp(R_E_mids, ambe_E, ambe_N)
#         phi0_counts = phi0_counts.reshape(1, -1)
#     elif sim_type == SimType.DDFUSION:  # D-D fusion
#         phi0_counts = None

#     if phi0_counts is None:
#         phi0 = None
#     else:
#         phi0 = NDHistogram(phi0_counts, [reduced_L_mids, R_E_mids])
# else:
#     phi0 = None

In [ ]:
for energy, eng_data in exp_data.items():
    print(energy)
    N = eng_data["N"]
    phi, unfold_info = unfold_spectrum(
        R,
        N,
        L_cut=0.05,
        # tolerance=0.0000001,
        max_iterations=1000,
        full_info=True
    )
    eng_data["phi"] = phi
    eng_data["unfold_info"] = unfold_info

In [ ]:
for energy, eng_data in exp_data.items():
    errors = eng_data["unfold_info"]["errors"]
    iters = len(errors)
    print(f"{energy}: {iters} iters.")

### Uncertainty Estimation

#### Horizontal

In [ ]:
sigma = 0.050   # MeVee

In [ ]:
lshift_R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    Path("response_matrix_R4_mono"),
    # min_L=-3*sigma,
    min_L=0,
    max_L=bins_max-3*sigma,
    L_bin_widths=bins_width
)
for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    lshift_mids = N.midpoints[0] - 3*sigma
    nonzero_mask = lshift_mids > 0
    lshift_N = NDHistogram(N.counts[nonzero_mask], [lshift_mids[nonzero_mask], N.midpoints[1]])
    # lshift_N = NDHistogram(N.counts, [lshift_mids, N.midpoints[1]])

    if np.isclose(lshift_N.midpoints[0], lshift_R.midpoints[0]).all():
        lshift_N = NDHistogram(lshift_N.counts, [lshift_R.midpoints[0], lshift_N.midpoints[1]])
    else:
        raise ValueError("R and N midpoints don't match")

    lshift_phi, *_ = unfold_spectrum(
        lshift_R,
        lshift_N,
        max_iterations=2000
    )
    eng_data["lshift_N"] = lshift_N
    eng_data["lshift_phi"] = lshift_phi

In [ ]:
rshift_R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    Path("response_matrix_R4_mono"),
    min_L=bins_min+3*sigma,
    max_L=bins_max+3*sigma,
    L_bin_widths=bins_width
)

for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    rshift_mids = N.midpoints[0] + 3*sigma
    rshift_N = NDHistogram(N.counts, [rshift_mids, N.midpoints[1]])

    if np.isclose(rshift_N.midpoints[0], rshift_R.midpoints[0]).all():
        rshift_N = NDHistogram(rshift_N.counts, [rshift_R.midpoints[0], rshift_N.midpoints[1]])
    else:
        raise ValueError("R and N midpoints don't match")

    rshift_phi, *_ = unfold_spectrum(
        rshift_R,
        rshift_N,
        max_iterations=2000
    )
    eng_data["rshift_N"] = rshift_N
    eng_data["rshift_phi"] = rshift_phi

### Lowpass Filtering

In [ ]:
from scipy.signal import butter, lfilter, filtfilt


def butter_lowpass(cutoff, fs, order=5):
    return butter(order, cutoff, fs=fs, btype='low', analog=False)


def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    # y = lfilter(b, a, data)
    y = filtfilt(b, a, data)
    return y


order = 6
period = 0.062
fs = 1/period
cutoff = fs/4

for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts)
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts)
    lmids = lshift_phi.midpoints[1]
    wrong_side_mask = lmids <= energy_num
    wrong_side_mask = wrong_side_mask.reshape((1, -1))
    rshift_phi_counts[wrong_side_mask] = 0
    
    lshift_counts_filtered = butter_lowpass_filter(lshift_phi_counts, cutoff, fs, order)
    rshift_counts_filtered = butter_lowpass_filter(rshift_phi_counts, cutoff, fs, order)
    eng_data["lshift_phi_filtered"] = NDHistogram(lshift_counts_filtered, lshift_phi.midpoints)
    eng_data["rshift_phi_filtered"] = NDHistogram(rshift_counts_filtered, rshift_phi.midpoints)

### Peak Finding

In [ ]:
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    if energy_num in [2.356, 2.542, 2.728]:
        continue
    # print(energy)
    phi = eng_data["phi"]
    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    phi_peak_idx, *_ = find_peaks(phi.counts.reshape(-1), prominence=0.01)
    lshift_peak_idx, *_ = find_peaks(lshift_phi_filtered.counts.reshape(-1), prominence=0.01)
    rshift_peak_idx, *_ = find_peaks(rshift_phi_filtered.counts.reshape(-1), prominence=0.01)
    # print(phi_peak_idx)
    # print([phi.midpoints[1][i] for i in phi_peak_idx])
    # print([lshift_phi_filtered.midpoints[1][i] for i in lshift_peak_idx])
    # print([rshift_phi_filtered.midpoints[1][i] for i in rshift_peak_idx])
    lo_error = energy_num - lshift_phi_filtered.midpoints[1][lshift_peak_idx[0]]
    hi_error = rshift_phi_filtered.midpoints[1][rshift_peak_idx[0]] - energy_num
    e_error = (float(lo_error), float(hi_error))
    print(energy_num)
    print(e_error)
    eng_data["e_error"] = e_error

In [ ]:
all_errorbars = []
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    if energy_num in [2.356, 2.542, 2.728]:
        continue
    e_error = eng_data["e_error"]
    all_errorbars.append((energy_num, e_error))
all_errorbars

### Plotting (Diagnostic)

In [ ]:
# exp_data["3.348_MeV"]["N"].counts

In [ ]:
figsize = (15,6)
# N (with L and R shift versions)
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    # ignore values we don't want
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    print(energy)
    N = eng_data["N"]
    lshift_N = eng_data["lshift_N"]
    rshift_N = eng_data["rshift_N"]

    phi = eng_data["phi"]
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts.reshape(-1))
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts.reshape(-1))
    lmids = lshift_phi.midpoints[1]
    energy_num = float(energy[:5])
    wrong_side_mask = lmids <= energy_num
    rshift_phi_counts[wrong_side_mask] = 0

    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    fig, axs = plt.subplots(1, 3, figsize=figsize)
    ax1, ax2, ax3 = axs
    
    ax1.plot(N.midpoints[0], N.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax1.plot(lshift_N.midpoints[0], lshift_N.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax1.plot(rshift_N.midpoints[0], rshift_N.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax1.set(xlabel="L (MeVee)", ylabel="Counts", title="PHD", yscale="symlog", ylim=(1, 1e4))
    ax1.legend()

    ax2.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax2.plot(lshift_phi.midpoints[1], lshift_phi_counts, linestyle="dashed", label="Left shift")
    ax2.plot(rshift_phi.midpoints[1], rshift_phi_counts, linestyle="dashed", label="Right shift")
    ax2.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (No Filter)"
    )
    ax2.legend()

    ax3.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax3.plot(lshift_phi_filtered.midpoints[1], lshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax3.plot(rshift_phi_filtered.midpoints[1], rshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax3.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (Filtered)"
    )
    ax3.legend()

    fig.suptitle(energy)

    plt.show()

In [ ]:
figsize = (15,6)
# N (with L and R shift versions)
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    # ignore values we don't want
    if energy_num > 3.286 or energy_num < 1.786:
        continue
    print(energy)
    N = eng_data["N"]
    lshift_N = eng_data["lshift_N"]
    rshift_N = eng_data["rshift_N"]

    phi = eng_data["phi"]
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts.reshape(-1))
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts.reshape(-1))
    lmids = lshift_phi.midpoints[1]
    energy_num = float(energy[:5])
    wrong_side_mask = lmids <= energy_num
    rshift_phi_counts[wrong_side_mask] = 0

    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    fig, axs = plt.subplots(1, 3, figsize=figsize)
    ax1, ax2, ax3 = axs
    
    ax1.plot(N.midpoints[0], N.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax1.plot(lshift_N.midpoints[0], lshift_N.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax1.plot(rshift_N.midpoints[0], rshift_N.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax1.set(xlabel="L (MeVee)", ylabel="Counts", title="PHD", yscale="symlog", ylim=(1, 1e4))
    ax1.legend()

    ax2.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax2.plot(lshift_phi.midpoints[1], lshift_phi_counts, linestyle="dashed", label="Left shift")
    ax2.plot(rshift_phi.midpoints[1], rshift_phi_counts, linestyle="dashed", label="Right shift")
    ax2.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (No Filter)"
    )
    ax2.legend()

    ax3.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax3.plot(lshift_phi_filtered.midpoints[1], lshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax3.plot(rshift_phi_filtered.midpoints[1], rshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax3.set(
        xlabel="E (MeV)", ylabel="Counts",
        # ylim=(0,0.2),
        title="Phi (Filtered)"
    )
    ax3.legend()

    fig.suptitle(energy)

    plt.show()

### Peak Finding

In [ ]:
for energy, eng_data in exp_data.items():
    # print(energy)
    phi = eng_data["phi"]
    phi_flat, phi_mids = get_graphable_data(phi)
    peaks, p_data = find_peaks(
        phi_flat,
        prominence=np.nanmax(phi_flat) / 100
    )
    # print(peaks)
    if len(peaks) > 0:
        idx_peak = phi_flat[peaks].argmax()
        peak_e = phi_mids[peaks][idx_peak]
    else:
        peak_e = None
    # print(peak_e)
    eng_data["peak_e"] = peak_e

In [ ]:
for energy, eng_data in exp_data.items():
    lshift_phi = eng_data["lshift_phi"]
    lshift_phi_flat, lshift_phi_mids = get_graphable_data(lshift_phi)
    lpeaks, lp_data = find_peaks(
        lshift_phi_flat,
        prominence=np.nanmax(lshift_phi_flat) / 100
    )
    if len(lpeaks) > 0:
        # print(energy)
        # print(lshift_phi_flat[lpeaks])
        # print(lshift_phi_mids[lpeaks])
        peak_e = eng_data["peak_e"]
        if peak_e is not None:
            low_mask = lshift_phi_mids[lpeaks] < peak_e
            masked_phi_flat = lshift_phi_flat[lpeaks][low_mask]
            if len(masked_phi_flat) > 0:
                idx_peak = masked_phi_flat.argmax()
                lshift_peak_e = lshift_phi_mids[lpeaks][low_mask][idx_peak]
            else:
                lshift_peak_e = None
        else:
            lshift_peak_e = None
    else:
        lshift_peak_e = None
    eng_data["lshift_peak_e"] = lshift_peak_e

In [ ]:
for energy, eng_data in exp_data.items():
    rshift_phi = eng_data["rshift_phi"]
    rshift_phi_flat, rshift_phi_mids = get_graphable_data(rshift_phi)
    rpeaks, rp_data = find_peaks(
        rshift_phi_flat,
        prominence=np.nanmax(rshift_phi_flat) / 100
    )
    if len(rpeaks) > 0:
        peak_e = eng_data["peak_e"]
        if peak_e is not None:
            high_mask = rshift_phi_mids[rpeaks] > peak_e
            # print(energy)
            # print(rshift_phi_mids[rpeaks][high_mask])
            # print(rshift_phi_flat[rpeaks][high_mask])
            masked_phi_flat = rshift_phi_flat[rpeaks][high_mask]
            if len(masked_phi_flat) > 0:
                idx_peak = masked_phi_flat.argmax()
                rshift_peak_e = rshift_phi_mids[rpeaks][high_mask][idx_peak]
            else:
                rshift_peak_e = None
        else:
            rshift_peak_e = None
    else:
        rshift_peak_e = None
    eng_data["rshift_peak_e"] = rshift_peak_e

In [ ]:
uncertainties = []
for energy, eng_data in exp_data.items():
    peak_e = eng_data["peak_e"]
    lshift_peak_e = eng_data["lshift_peak_e"]
    rshift_peak_e = eng_data["rshift_peak_e"]
    uncert_lo = None if lshift_peak_e is None or peak_e is None else float(peak_e - lshift_peak_e)
    uncert_hi = None if rshift_peak_e is None or peak_e is None else float(rshift_peak_e - peak_e)
    e_uncert = uncert_lo, uncert_hi
    # print(f"{energy}: ({lshift_peak_e}, {peak_e}, {rshift_peak_e}) -> {e_uncert}")
    uncertainties.append((float(peak_e) if peak_e is not None else None, uncert_lo, uncert_hi))
    eng_data["e_uncertianty"] = e_uncert
uncertainties

## Plotting

In [ ]:
figsize = (9, 6)
fontsize = 20

In [ ]:
plot_data = [(float(energy[:4]), eng_data["e_uncertianty"][0], eng_data["e_uncertianty"][1]) for energy, eng_data in exp_data.items()]
energies, lo_uncerts, hi_uncerts = zip(*plot_data)
energies = list(energies)[2:]
lo_uncerts = list(lo_uncerts)[2:]
hi_uncerts = list(hi_uncerts)[2:]

fig, ax = plt.subplots(figsize=figsize)
ax.plot(energies, lo_uncerts, marker="o", markersize=3, label="Lo")
ax.plot(energies, hi_uncerts, marker="o", markersize=3, label="Hi")
ax.set(xlabel="Energy (MeV)", ylabel="Uncertainty (MeV)")
ax.legend()
plt.show()

In [ ]:
energy = "4.5000MeV"
eng_data = exp_data[energy]
phi = eng_data["phi"]
lshift_phi = eng_data["lshift_phi"]
rshift_phi = eng_data["rshift_phi"]
phi_flat, phi_mids = get_graphable_data(phi)
l_phi_flat, l_phi_mids = get_graphable_data(lshift_phi)
r_phi_flat, r_phi_mids = get_graphable_data(rshift_phi)

fig, ax = plt.subplots(figsize=figsize)
ax.plot(phi_mids, phi_flat, marker="o", markersize=3, label="Normal")
ax.plot(l_phi_mids, l_phi_flat, marker="o", markersize=3, label="Lo")
ax.plot(r_phi_mids, r_phi_flat, marker="o", markersize=3, label="Hi")
ax.set(xlabel="Energy (MeV)", ylabel="Uncertainty (MeV)")
ax.legend()
plt.show()

### Standard

In [ ]:

fig, ax = plt.subplots(figsize=figsize)
ax.plot(range(len(chis)), chis)
ax.set(yscale="log", ylabel="Chi^2/n", xlabel="Iteration", title="Stopping Criteria")
plt.show()

In [ ]:
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.plot(phi_mids, phi_flat, marker="o", markersize=6, label="Unfolded from simulated light output")
if sim_type == SimType.AMBE:
    ax.plot(ambe_E, ambe_N * ambe_scaling, marker="o", markersize=6, label="AmBe ISO standard")
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
plt.show()

### Uncertainty - Horizontal

In [ ]:
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.plot(phi_mids, phi_flat, marker="o", markersize=6, label="Unfolded from simulated light output")
ax.plot(lshift_phi_mids, lshift_phi_flat, marker="o", markersize=6, label="Left-shifted")
ax.plot(rshift_phi_mids, rshift_phi_flat, marker="o", markersize=6, label="Right-shifted")
if sim_type == SimType.AMBE:
    ax.plot(ambe_E, ambe_N * ambe_scaling, marker="o", markersize=6, label="AmBe ISO standard")
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
ax.legend()
plt.show()

### Animations

In [ ]:
phis = unfold_info["phis"]
phis_flat = [phi.counts.reshape(-1) for phi in phis]

In [ ]:
print(f"You have {len(errors)} frames to animate.")
showevery = helpers.get_input_with_default("Enter n for 'Show every n'th frame', or press Enter for default (1) >",
                                           1, int)

In [ ]:
import matplotlib.animation as animation

phis_to_show = phis_flat[showevery-1::showevery]
plot_max = np.nanmax(phis_to_show[-1])

with plt.ioff():
    fig, ax = plt.subplots(figsize=figsize)
    phi_plot = ax.plot(phi_mids, phis_to_show[0], marker="o", markersize=6)[0]
    plot_frame = ax.annotate(f"Frame {showevery}", (1, 1), xycoords="axes fraction", horizontalalignment="right", verticalalignment="top")
    if sim_type == 2:
        ax.plot(ambe_E, ambe_N * ambe_scaling, marker="o", markersize=6)
    # ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
    # if len(peaks) > 0:
    #     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
    #     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
    #         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
    ax.set(
        xlabel="E (MeV)",
        ylabel="Counts",
        # ylim=(0, 0.05),
        ylim=(0, plot_max * 1.1),
        title="Unfolded Spectrum (Simulated AmBe Neutrons)")
    
    def update(frame):
        y = phis_to_show[frame]
        phi_plot.set_ydata(y)
        plot_frame.set_text(f"Frame {(frame + 1)*showevery}")
        return (phi_plot,plot_frame)
    
    ani = animation.FuncAnimation(fig=fig, func=update, frames=len(phis_to_show), interval=120)
    ani.save("ambe_phis.gif")

In [ ]:
weights = unfold_info["weights"]
W_L_mids, W_phi_mids = weights[0].midpoints
print(W_L_mids.shape)
print(W_phi_mids.shape)
print(weights[0].shape)

L_widths = (W_L_mids[1:] - W_L_mids[:-1])
L_widths = np.insert(L_widths, (0,), L_widths[0])
phi_widths = W_phi_mids[1:] - W_phi_mids[:-1]
phi_widths = np.insert(phi_widths, (0,), phi_widths[0])

_x = W_phi_mids - (phi_widths / 2)
_y = W_L_mids - (L_widths / 2)
_xx, _yy = np.meshgrid(_x, _y)
x, y = np.ravel(_xx), np.ravel(_yy)

_xxw, _yyw = np.meshgrid(phi_widths, L_widths)
xw, yw = np.ravel(_xxw), np.ravel(_yyw)

tops = [np.ravel(weight.counts) for weight in weights]
bottom = np.zeros_like(tops[0])

Zmax_list = [np.nanmax(top) for top in tops]
Zmax = max(Zmax_list)
Zmin = 0

In [ ]:
import matplotlib as mpl

vaporwave_colors = [
    [255, 255, 255],
    [128, 69, 229],
    [75,127,255],
    [0,255,255],
    [255,186,129],
    [255,209,86],
    [252,120,183]
]
vaporwave_colors = [[value/255 for value in color] for color in vaporwave_colors]
vaporwave = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", vaporwave_colors, N=256)
# vaporwave = vaporwave.resampled(256)
vaporwave

In [ ]:
norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
sc.set_array([])

weights_colors = [vaporwave(norm(top)) for top in tops]

In [ ]:
print(f"You have {len(errors)} frames to animate.")
showevery = helpers.get_input_with_default("Enter n for 'Show every n'th frame', or press Enter for default (1) >",
                                           1, int)

In [ ]:
# fig, ax = plt.subplots(figsize=figsize, projection="3d")
with plt.ioff():
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection="3d")
    
    artists = []
    _frame = 0
    for i in range(showevery-1, len(tops), showevery):
        print(f"{_frame % 10}", end="")
        _frame += 1
        top = tops[i]
        color = weights_colors[i]
        nan_mask = ~np.isnan(top)
        masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
        masked_color = color[nan_mask]
        # W_plot = ax.bar3d(x, y, bottom, xw, yw, top, color=color)
        W_plot = ax.bar3d(*masked_args, color=masked_color)
        frame = ax.annotate(f"Frame {i+1}", (1, 1), xycoords="axes fraction", horizontalalignment="right", verticalalignment="top")
        artists.append([W_plot, frame])

In [ ]:
# print([type(x) for x in artists])
ax.set(xlabel="Neutron energy (MeV)", ylabel="Light output (MeVee)", zlabel="Weight")
ani = animation.ArtistAnimation(fig=fig, artists=artists, interval=120)
ani.save("ambe_weights.gif")
# plt.show()

## End